# S3 en direct : sous-titres français pendant que l'audio joue

Un fichier audio arabe est traité comme un flux : toutes les secondes, le segment en cours est transcrit par S3 (encodeur Cohere → **projecteur** → Qwen2.5-1.5B-Instruct), puis traduit en français par le même LLM. Chaque sous-titre garde l'instant où il serait apparu en direct (temps de calcul compris), puis un lecteur rejoue l'audio en affichant chaque sous-titre à cet instant.

- **Blanc** : mots confirmés (même mot à la même place deux passes de suite, il ne changera plus). **Gris** : provisoire.
- Un segment est clos après 0,5 s de silence (pas avant 2 s de parole), ou d'office à 15 s. Sa traduction est alors figée.
- `MODE = "secours_morceaux"` : rien n'est affiché avant la clôture, puis le segment entier d'un bloc (aucune réécriture, moins de calcul).
- C'est une cascade : la traduction directe depuis l'audio ne marche pas tant que le temps 2 n'est pas fait (voir `s3_tester_audio.ipynb`).
- Logique dans `code/direct/flux_s3.py`, réglages dans `configs/direct.yaml` (sections `direct` et `direct_s3`).

**Ne pas committer ce notebook avec ses sorties** si l'audio vient de MGB-2 (licence de recherche QCRI).

In [ ]:
# À modifier : chemin de votre fichier audio, ou None pour un extrait du test MGB-2
CHEMIN_AUDIO = None
DUREE_EXTRAIT_S = 60           # durée de l'extrait MGB-2 quand CHEMIN_AUDIO est None
MODE = "deux_passes"           # ou "secours_morceaux"

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import yaml

racine = Path.cwd()
while not (racine / "configs" / "s3.yaml").exists() and racine.parent != racine:
    racine = racine.parent
for dossier in ("projecteur", "ecoute", "donnees", "traduction", "mesures", "demo", "direct"):
    if str(racine / "code" / dossier) not in sys.path:
        sys.path.insert(0, str(racine / "code" / dossier))

from flux_s3 import FluxS3, reglages_direct_s3
from transcrire_s3 import TranscripteurS3, charger_audio, transcripteur_factice

ESSAI = os.environ.get("S3_NOTEBOOK_ESSAI") == "1"   # petits modèles factices (tests), sans GPU
reglages = reglages_direct_s3(MODE)
depart = time.perf_counter()
transcripteur = transcripteur_factice(reglages["section_s3"]) if ESSAI else TranscripteurS3.charger(reglages["section_s3"])
print(f"modèles chargés en {time.perf_counter() - depart:.0f} s")

## Choisir l'audio

In [ ]:
if ESSAI:
    signal = (0.1 * np.random.default_rng(0).standard_normal(20 * 16000)).astype(np.float32)
    signal[8 * 16000: 9 * 16000] *= 0.001   # une pause d'1 s
    origine = "bruit factice (mode essai)"
elif CHEMIN_AUDIO is not None:
    signal = charger_audio(CHEMIN_AUDIO)
    origine = str(CHEMIN_AUDIO)
else:
    from benchmark_traduction import lire_xml_mgb2
    config_encodage = yaml.safe_load((racine / "configs" / "encodage_s3.yaml").read_text(encoding="utf-8"))
    sources = config_encodage["sources"]
    racine_mgb2 = Path(sources["racine_mgb2"]) if Path(sources["racine_mgb2"]).is_absolute() else racine / sources["racine_mgb2"]
    with open(racine / config_encodage["sortie"]["dossier"] / "test.jsonl", encoding="utf-8") as fichier:
        premier = json.loads(fichier.readline())
    # Extrait calé sur des segments entiers de la référence (émission du test, jamais vue à l'entraînement).
    segments_xml = sorted(lire_xml_mgb2(racine_mgb2 / sources["sous_dossier_xml"] / f"{premier['episode']}.xml"),
                          key=lambda s: s["debut"])
    suite = [s for s in segments_xml if s["debut"] >= premier["debut"]]
    references = [s for s in suite if s["fin"] <= suite[0]["debut"] + DUREE_EXTRAIT_S] or suite[:1]
    debut, fin = references[0]["debut"], references[-1]["fin"]
    signal = charger_audio(racine_mgb2 / premier["wav"])[int(debut * 16000): int(fin * 16000)]
    origine = f"MGB-2 test, émission {premier['episode']}, {debut:.1f} à {fin:.1f} s"
print(origine, f": {len(signal) / 16000:.1f} s")

## Sous-titrage en direct

Le flux est d'abord calculé (une passe par seconde d'audio, avec l'horloge du direct : un calcul plus long qu'une seconde retarde la passe suivante). Puis le lecteur affiche chaque sous-titre quand la lecture atteint son instant d'apparition : l'affichage suit le bouton lecture, la pause et les retours en arrière, et montre le vrai retard du système.

Pourquoi pas un affichage en direct depuis Python : la cellule ne sait pas quand le lecteur démarre vraiment (lecture automatique bloquée, chargement de l'audio, clic), donc les deux horloges ne coïncident pas.

In [ ]:
from IPython.display import HTML, display

from flux_s3 import lecteur_html

flux = FluxS3(transcripteur, reglages)
depart = time.perf_counter()
messages = list(flux.diffuser(signal))
calcul = time.perf_counter() - depart
print(f"{len(messages)} affichages calculés en {calcul:.1f} s pour {len(signal) / 16000:.1f} s d'audio "
      f"({'plus' if calcul < len(signal) / 16000 else 'moins'} rapide que le temps réel)")
display(HTML(lecteur_html(signal, messages)))

## Bilan par segment

Retard = instant où la traduction du segment est figée − fin de la parole (début du silence de clôture). Taux de réécriture : part des mots affichés qui changent à la passe suivante (0 en mode secours, qui n'affiche rien de provisoire).

In [ ]:
for e in flux.evenements:
    print(f"{e['id_segment']} [{e['debut_s']:6.1f} - {e['fin_s']:6.1f} s] retard {e['affichage_definitif_s'] - e['fin_parole_s']:.1f} s, "
          f"{len(e['passes_provisoires'])} passes provisoires")
    print("  AR :", e["ar"])
    print("  FR :", e["texte_final"])
bilan = flux.bilan()
print(f"\n{bilan.get('nb_segments', 0)} segments, mode {MODE}")
if bilan.get("nb_segments"):
    print(f"retard moyen {bilan['retard_moyen_s']} s, max {bilan['retard_max_s']} s ; réécriture {bilan['taux_reecriture_pct']} %")